In [ ]:
#get parameters
import sys
sys.path.append('..')
from src.grass_functions import*
from project_info import *

#for project
Project_Area = 'nebraska_regression_stantec'
sr = '26852' #set to None if you want to use the DEM's original projection
res = '10m' #meters #DEM resolution, options are '1m', '3m', '10m', '30m', 'OPR'

#dem info turn into class later
dem_preprocessed = False
dem_base_name = 'state_dem' #for saving in grass
aligned = False
carved = True

#initiate class
project = ProjectInformation(Project_Area,sr,res,dem_preprocessed,dem_base_name,aligned,carved)
project.create_output_dirs()
initialize_grass_db(project.Location, project.Mapset, project.sr)
output_geo = []

for num in np.arange(1,76):
    #for run
    data_scale = 'pnt_id'
    analysis_scale = 'pnt_id'
    aoi = num
    geometry = 'point' #
    
    aoi = GrassWatershed(project, data_scale,analysis_scale,aoi,geometry)
    aoi.set_grass_selection()
    initialize_grass_db(project.Location, project.Mapset, project.sr)
    aoi.list_existing_grass(print_it=False)
    aoi.set_dem_name()
    aoi.assign_grass_variables()
    aoi.get_grass_grid_size()
    aoi.get_rough_watershed_data()
    aoi.get_basin_area()
    
    regression_data = RegressionData(project,aoi)
    regression_data.set_aoi()
    regression_data.get_drainage_area()
    regression_data.get_stream_order()
    regression_data.get_basin_shape()
    regression_data.CN_mean = regression_data.get_raster_avg(project.raster_dir/'NE_State_CN_nad.tif','CN_mean')
    regression_data.PRISMyr_mm =regression_data.get_raster_avg(project.raster_dir/'PRISM_yr_NE_Statewide.tif','PRISMyr_mm')
    regression_data.get_main_ch_slope()
    regression_data.get_basin_relief()
    gs.run_command('v.out.ogr', input=  aoi.v_basins ,type = 'area',output = aoi.basins, format = 'GeoJSON')
    
    #initialize
    regression_regions = project.vector_dir/'draft_regression.shp'
    regression_flows = RegressionEquations(project,aoi,regression_data,regression_regions)
    #regionalization
    regression_flows.get_regions()
    #calculate and add flows to geojson vector
    regression_flows.calc_flows()
    regression_flows.add_flows_to_outlet()
    output_geo.append(aoi.outlet)

/opt/conda/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.0
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Database Location Exists
Database Mapset Exists
None
{'GISDBASE': '/home/grassdata', 'LOCATION_NAME': 'nebraska_regression_stantec_26852', 'MAPSET': 'PERMANENT'}
base data is 1, analysis area is 1
Database Location Exists
Database Mapset Exists
None
{'GISDBASE': '/home/grassdata', 'LOCATION_NAME': 'nebraska_regression_stantec_26852', 'MAPSET': 'PERMANENT'}


Current GRASS GIS 7 environment:
/opt/conda/lib/python3.9/site-packages/geopandas/io/file.py:299: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,
Current GRASS GIS 7 environment:
Raster MASK removed


using existing project-wide watershed data for the project area


Check if OGR layer <aoi_1_outlet> contains polygons...
   0 100
         overwritten
Creating attribute table for layer <aoi_1_outlet>...
Importing 1 features (OGR layer <aoi_1_outlet>)...
   0 100
-----------------------------------------------------
Building topology for vector map <aoi_1_outlet_raw@PERMANENT>...
Registering primitives...
Input </home/data/Vectors/nebraska_regression_stantec/aoi_1_outlet.geojson>
successfully imported without reprojection
Reading raster map <r_streams_nebraska_regression_stantec_rough>...
   0   3   6   9  12  15  18  21  24  27  30  33  36  39  42  45  48  51  54  57  60  63  66  69  72  75  78  81  84  87  90  93  96  99 100
Reading raster map <accum_nebraska_regression_stantec_rough>...
   0   3   6   9  12  15  18  21  24  27  30  33  36  39  42  45  48  51  54  57  60  63  66  69  72  75  78  81  84  87  90  93  96  99 100
         overwritten
Building topology for vector map <v_outlet_1_rough@PERMANENT>...
Registering primitives...
All in RAM c

added basin aoi to grass


         overwritten
Buffering areas...
 100
Cleaning buffers...
Building parts of topology...
Building topology for vector map <aoi_1_aoi_raw_buffer@PERMANENT>...
Registering primitives...
Snapping boundaries...
Reading features...
Snap vertices Pass 1: select points
   0 100
Snap vertices Pass 2: assign anchor vertices
   4   9  14  19  24  29  34  39  44  49  54  59  64  69  74  79  84  89  94  99 100
Snap vertices Pass 3: snap to assigned points
   0 100
Breaking polygons...
Breaking polygons (pass 1: select break points)...
 100
Breaking polygons (pass 2: break at selected points)...
 100
Removing duplicates...
 100
Breaking boundaries...
   0 100
Removing duplicates...
 100
Cleaning boundaries at nodes
 100
Building topology for vector map <aoi_1_aoi_raw_buffer@PERMANENT>...
Building areas...
   0 100
Removing dangles...
 100
Removing bridges...
 100
Attaching islands...
Building topology for vector map <aoi_1_aoi_raw_buffer@PERMANENT>...
Attaching islands...
   0 100
Calculating